# Runbook 2.1.b: Threat actors and their motivations  (MODEL SOLUTION, instructor copy, do not distribute)

**Skill:** Analyze Risk · **Maps to:** 2.1.B · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** In this challenge, you will implement an automated log classifier for the security team at Meridian University. You are given a list of alert logs, each containing an ID and a text description. You must classify each alert to identify the most likely threat actor category responsible for the attack based on key phrases present in the logs.

**Lesson objective.** Identify the six major threat actor categories, explain what motivates each one, and use motivation to recommend the most appropriate defensive measure for a given attack scenario.

### What comes in
A list of log entries:
```python
[
    {"id": "ALT-01", "log": "Ransomware deployed. Files encrypted. Payment of 2.5 BTC demanded."},
    {"id": "ALT-02", "log": "FTP scan detected. Automated port scan looking for vulnerabilities."}
]
```

### What must come out
A dictionary mapping log IDs to their threat actor classification:
```python
{
    "ALT-01": "Cybercriminal",
    "ALT-02": "Script Kiddie"
}
```

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.


### Rules to respect
- [ ] Deterministic, offline, standard library only.
- [ ] Must pass all cases in the tests.
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or composite) and must pass the tests unchanged.

#### Entry point
```python
def solve(logs: list) -> dict:
```

#### Input (`logs`)
A list of dictionaries, where each dictionary represents an alert log:
```python
[
    {"id": "ALT-101", "log": "Detected a zero-day exploit and quiet rootkit installation targeting research databases."},
    {"id": "ALT-102", "log": "Attacker ran an automated scan looking for default passwords on port 80."}
]
```

#### Output (`result`)
A dictionary mapping each alert ID to its classified threat actor category:
```python
{
    "ALT-101": "Nation-State / APT",
    "ALT-102": "Script Kiddie"
}
```

#### Guarantees / rules
- Pure function, deterministic, no I/O, no network.
- Standard library only.
- Categorization is based on substring matches (case-insensitive) in the `log` text:
  - **Nation-State / APT**: `"zero-day"`, `"state-sponsored"`, `"espionage"`, `"sabotage"`, `"rootkit"`, `"lateral movement"`, `"exfiltrat"`
  - **Cybercriminal**: `"ransomware"`, `"bitcoin"`, `"payment"`, `"extortion"`, `"payroll"`, `"fraud"`, `"financial"`
  - **Hacktivist**: `"protest"`, `"deface"`, `"ideolog"`, `"political"`, `"publicity"`, `"equity"`
  - **Insider Threat**: `"employee"`, `"disgruntled"`, `"exit interview"`, `"insider"`, `"negligent"`, `"usb"`
  - **Script Kiddie**: `"script kiddie"`, `"coin-miner"`, `"coinminer"`, `"default password"`, `"port scan"`, `"ftp exploit"`, `"automated scan"`
- If multiple categories match a log, prioritize the classification in this order:
  1. `"Nation-State / APT"`
  2. `"Cybercriminal"`
  3. `"Hacktivist"`
  4. `"Insider Threat"`
  5. `"Script Kiddie"`
- If no keywords match, the classification should be `"Unknown"`.

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

This is the model solution.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
from typing import List, Dict, Any

CATEGORIES = [
    ("Nation-State / APT", ["zero-day", "state-sponsored", "espionage", "sabotage", "rootkit", "lateral movement", "exfiltrat"]),
    ("Cybercriminal", ["ransomware", "bitcoin", "payment", "extortion", "payroll", "fraud", "financial"]),
    ("Hacktivist", ["protest", "deface", "ideolog", "political", "publicity", "equity"]),
    ("Insider Threat", ["employee", "disgruntled", "exit interview", "insider", "negligent", "usb"]),
    ("Script Kiddie", ["script kiddie", "coin-miner", "coinminer", "default password", "port scan", "ftp exploit", "automated scan"]),
]

def solve(logs: List[Dict[str, Any]]) -> Dict[str, str]:
    res = {}
    for entry in logs:
        lid = entry["id"]
        text = entry.get("log", "").lower()
        classified = "Unknown"
        
        for cat_name, keywords in CATEGORIES:
            if any(kw in text for kw in keywords):
                classified = cat_name
                break
        res[lid] = classified
    return res


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

def test_basic_script_kiddie():
    logs = [{"id": "A", "log": "Detected automated scan on port 21. FTP exploit attempt."}]
    res = impl.solve(logs)
    assert res == {"A": "Script Kiddie"}

def test_basic_cybercriminal():
    logs = [{"id": "B", "log": "Ransomware locked user database. Send payment to local address."}]
    res = impl.solve(logs)
    assert res == {"B": "Cybercriminal"}

def test_basic_hacktivist():
    logs = [{"id": "C", "log": "Main landing page was defaced in protest of policy."}]
    res = impl.solve(logs)
    assert res == {"C": "Hacktivist"}

def test_basic_insider():
    logs = [{"id": "D", "log": "Disgruntled employee suspected of downloading sensitive files."}]
    res = impl.solve(logs)
    assert res == {"D": "Insider Threat"}

def test_basic_nation_state():
    logs = [{"id": "E", "log": "Quiet rootkit installation detected using zero-day exploit."}]
    res = impl.solve(logs)
    assert res == {"E": "Nation-State / APT"}

def test_priority_conflict():
    logs = [{"id": "F", "log": "Zero-day exploit was deployed during a noisy port scan."}]
    res = impl.solve(logs)
    assert res == {"F": "Nation-State / APT"}
    
    logs = [{"id": "G", "log": "Ransomware deployed during political protest."}]
    res = impl.solve(logs)
    assert res == {"G": "Cybercriminal"}

def test_unknown_log():
    logs = [{"id": "H", "log": "Normal login attempt succeeded."}]
    res = impl.solve(logs)
    assert res == {"H": "Unknown"}

def test_case_insensitive():
    logs = [{"id": "I", "log": "EXFILTRATION OF HR DATABASE TO EXTERNAL IP ADDRESS"}]
    res = impl.solve(logs)
    assert res == {"I": "Nation-State / APT"}

def test_empty_logs():
    assert impl.solve([]) == {}

def test_multiple_logs():
    logs = [
        {"id": "1", "log": "Zero-day rootkit staged."},
        {"id": "2", "log": "A disgruntled worker exit interview logged."},
        {"id": "3", "log": "Anonymous post criticizing company on public forum defaced website."}
    ]
    res = impl.solve(logs)
    assert res == {
        "1": "Nation-State / APT",
        "2": "Insider Threat",
        "3": "Hacktivist"
    }


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Run your finished tool on the examples from the brief, then change one value and predict the answer before you run it again.

- [ ] Every example says OK
- [ ] Your own changed example gives the answer you predicted

In [ ]:
checks = [
    (lambda: solve([]),
     {}),
]

for n, (run, want) in enumerate(checks, 1):
    try:
        got = run()
    except Exception as e:
        got = "error: " + repr(e)
    print("example", n, "OK  " if got == want else "DIFF", "got:", got, " expected:", want)


## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it